In [12]:
import pymysql
from pymysql import Error

# ===================== Database Configuration =====================
DB_CONFIG = {
    "host": "localhost",
    "user": "root",
    "password": "003613",
    "database": "comp7640_ecommerce",
    "charset": "utf8mb4"
}

# ===================== Product Search Function =====================
def search_products_by_tag(search_keyword):
    connection = None
    cursor = None
    results = []
    
    try:
        connection = pymysql.connect(**DB_CONFIG)
        cursor = connection.cursor(pymysql.cursors.DictCursor)

        sql = """
        SELECT 
            p.product_id,
            p.name,
            p.vendor_id,
            p.listed_price,
            p.stock_quantity,
            p.tag1,
            p.tag2,
            p.tag3,
            v.business_name
        FROM products p
        JOIN vendors v ON p.vendor_id = v.vendor_id
        WHERE 
            p.name LIKE %s
            OR p.tag1 LIKE %s
            OR p.tag2 LIKE %s
            OR p.tag3 LIKE %s
        ORDER BY p.product_id;
        """
        
        match = f"%{search_keyword}%"
        cursor.execute(sql, (match, match, match, match))
        results = cursor.fetchall()

    except Error as e:
        print(f"Error: {e}")
    finally:
        if cursor:
            cursor.close()
        if connection:
            connection.close()
            
    return results

# ===================== Display Results with Perfect Alignment =====================
def display_products(products):
    if not products:
        print("No products found.")
        return

    print("\n" + "=" * 150)
    fmt = "{:<10} {:<22} {:<10} {:<12} {:<14} {:<13} {:<13} {:<13} {:<15}"
    print(fmt.format(
        "product_id", "name", "vendor_id", "listed_price",
        "stock_quantity", "tag1", "tag2", "tag3", "business_name"
    ))
    print("-" * 150)

    for p in products:
        tag1 = p['tag1'] if p['tag1'] else "NULL"
        tag2 = p['tag2'] if p['tag2'] else "NULL"
        tag3 = p['tag3'] if p['tag3'] else "NULL"

        print(fmt.format(
            p['product_id'],
            p['name'][:20],
            p['vendor_id'],
            f"{p['listed_price']:.2f}",
            p['stock_quantity'],
            tag1,
            tag2,
            tag3,
            p['business_name']
        ))
    
    print("=" * 150)

# ===================== Continuous Search + Exit Function =====================
if __name__ == "__main__":
    print("===== E-Commerce Product Discovery =====")
    print("⚠️  Type 'exit' to quit the program\n")
    
    while True:
        keyword = input("Search keyword (or 'exit' to quit): ").strip()

        # Exit function
        if keyword.lower() == "exit":
            print("\nProgram exited successfully.")
            break
        
        if not keyword:
            print("Error: Keyword cannot be empty!\n")
            continue

        # Execute search
        products = search_products_by_tag(keyword)
        display_products(products)

===== E-Commerce Product Discovery =====
⚠️  Type 'exit' to quit the program



Search keyword (or 'exit' to quit):  camera



product_id name                   vendor_id  listed_price stock_quantity tag1          tag2          tag3          business_name  
------------------------------------------------------------------------------------------------------------------------------------------------------
5          alpha 3                3          50000.00     10000          electronics   camera        kitchen       SONY           
6          Stainless Steel Kett   3          199.00       60             electronics   camera        appliance     SONY           


Search keyword (or 'exit' to quit):  phone



product_id name                   vendor_id  listed_price stock_quantity tag1          tag2          tag3          business_name  
------------------------------------------------------------------------------------------------------------------------------------------------------
1          IPhone 17 Pro          1          9999.00      9000           electronics   phone         wireless      Apple          


Search keyword (or 'exit' to quit):  exit



Program exited successfully.
